In [7]:
#Very simple NN. Raw audio -> w2v -> FCC ->  classification


In [8]:
import pandas as pd

df = pd.read_csv('SEP-28k-Filtered_with_Split_and_Path.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28177 entries, 0 to 28176
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        28177 non-null  int64 
 1   Show              28177 non-null  object
 2   EpId              28177 non-null  int64 
 3   ClipId            28177 non-null  int64 
 4   SEP28k-T          28177 non-null  object
 5   SEP28k-D          28177 non-null  object
 6   Prolongation      28177 non-null  int64 
 7   Block             28177 non-null  int64 
 8   SoundRep          28177 non-null  int64 
 9   WordRep           28177 non-null  int64 
 10  Interjection      28177 non-null  int64 
 11  NoStutteredWords  28177 non-null  int64 
 12  filepath          28177 non-null  object
dtypes: int64(9), object(4)
memory usage: 2.8+ MB


In [9]:
#Split entries based on SEP28k-T SEP
#copy so that they're their own objects
T_train_df = df[df["SEP28k-T"]== 'train'].copy()
T_dev_df = df[df["SEP28k-T"]== 'dev'].copy()
T_test_df = df[df["SEP28k-T"]== 'test'].copy()

In [10]:
# pytorch setup
import torch
import torch.nn as nn
import torchaudio
import soundfile as sf
from torch.utils.data import Dataset, DataLoader

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
w2v_starter = torchaudio.pipelines.WAV2VEC2_BASE
w2v_model = w2v_starter.get_model().to(device)

In [14]:
embedded_audio_files_Train = []
embedded_audio_files_Dev = []
embedded_audio_files_Test = []


In [ ]:
def raw_audio_to_w2v (filename):
    raw_waveform_nd, raw_sample_rate = sf.read(filename) 
    raw_waveform = torch.from_numpy(raw_waveform_nd)
    raw_waveform = raw_waveform.float() #w2v expects 32bit float
    if raw_sample_rate != 16000: #makes sure audio is 16khz
        wav = torchaudio.functional.resample(wav, raw_sample_rate, 16000)
    raw_waveform = raw_waveform.unsqueeze(0) #w2v expects at least 2d tensor, first dimension is batch size, so have to unsqueeze to (1, 48000) for single file.
    raw_waveform = raw_waveform.to(device)
    embedded_output, _ = w2v_model(raw_waveform)
    return embedded_output

In [11]:
#Create a dataset for audio files and labels (based on Mohapatra et al.)
class SEP_DataSet (Dataset):
    def __init__(self, features, labels):
        super().__init__()
        self.features = features
        self.labels = labels

    def __getitem__(self,index) :
        return self.features[index], self.labels[index]
